In [1]:
import os
import json
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm import tqdm
import stanza

# ===== PATH SETUP =====
BASE_DIR = "../JCDL_Code_2022_2025/Scientific_Novelty_Detection_2022_2025"
CACHE_DIR = os.path.join(BASE_DIR, "cache")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
TRIPLET_DIR = os.path.join(BASE_DIR, "Triplets", "Blogs")
TEMP_DIR = os.path.join(BASE_DIR, "temp_pdf")

os.makedirs(TRIPLET_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

TASKS = ["Dia", "MT", "NLI", "Par", "QA", "SA", "Sum"]

GROBID_URL = "http://localhost:8070/api/processFulltextDocument"

# ===== STANZA =====
nlp = stanza.Pipeline("en", processors="tokenize,pos,lemma", use_gpu=False, tokenize_batch_size=32)

2026-02-27 20:59:12 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-02-27 20:59:15 INFO: Downloaded file to C:\Users\spars\stanza_resources\resources.json
2026-02-27 20:59:15 WARNING: Language en package default expects mwt, which has been added
2026-02-27 20:59:15 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-02-27 20:59:15 INFO: Using device: cpu
2026-02-27 20:59:15 INFO: Loading: tokenize
2026-02-27 20:59:19 INFO: Loading: mwt
2026-02-27 20:59:19 INFO: Loading: pos
2026-02-27 20:59:22 INFO: Loading: lemma
2026-02-27 20:59:22 INFO: Done loading processors!


In [2]:
def load_blogs_metadata(task):
    path = os.path.join(CACHE_DIR, f"Blogs_{task}_metadata.json")
    with open(path, "r") as f:
        return json.load(f)

In [3]:
def download_pdf(url, save_path):
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            with open(save_path, "wb") as f:
                f.write(r.content)
            return True
    except Exception:
        pass
    return False

In [4]:
def parse_with_grobid(pdf_path):
    with open(pdf_path, "rb") as f:
        r = requests.post(
            GROBID_URL,
            files={"input": f}
        )
    if r.status_code == 200:
        return r.text
    return None

In [5]:
def extract_text_from_tei(tei_xml):
    try:
        root = ET.fromstring(tei_xml)
    except:
        return ""

    texts = []
    for elem in root.iter():
        if elem.text:
            cleaned = elem.text.strip()
            if cleaned:
                texts.append(cleaned)

    return " ".join(texts)

In [6]:
def extract_sentences_chunked(text, chunk_size=3000):

    sentences = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]

        try:
            doc = nlp(chunk)
            for sent in doc.sentences:
                sentence_text = " ".join([token.text for token in sent.tokens])
                sentences.append(sentence_text)
        except:
            continue

    return sentences

In [7]:
def extract_triplets_from_sentences(sentences, paper_id, task, year):

    rows = []

    for idx, sentence in enumerate(sentences):

        words = sentence.split()

        if len(words) >= 3:
            sub = words[0]
            pred = words[1]
            obj = " ".join(words[2:5])

            rows.append({
                "topic": task,
                "paper_ID": paper_id,
                "sentence_ID": idx,
                "info-unit": "auto",
                "sub": sub,
                "pred": pred,
                "obj": obj,
                "triplets": f"{sub} {pred} {obj}",
                "pred_weights": None,
                "year": year
            })

    return rows

In [8]:
def build_blogs_triplets(task):

    metadata_path = os.path.join(CACHE_DIR, f"Blogs_{task}_metadata.json")
    output_path = os.path.join(TRIPLET_DIR, f"{task}_triplets_results.csv")
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"Blogs_{task}_triplet_checkpoint.json")

    # ==========================
    # Skip if already completed
    # ==========================
    if os.path.exists(output_path) and not os.path.exists(checkpoint_path):
        print(f"{task} Blogs triplets already completed.")
        return

    # ==========================
    # Load metadata
    # ==========================
    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    # ==========================
    # Resume if checkpoint exists
    # ==========================
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            checkpoint = json.load(f)
        start_idx = checkpoint["index"]
        print(f"Resuming {task} from index {start_idx}")
    else:
        start_idx = 0

    # ==========================
    # Main Loop
    # ==========================
    for i in tqdm(range(start_idx, len(metadata))):

        paper = metadata[i]
        paper_id = paper["id"].split("/")[-1]
        year = paper.get("year")
        pdf_url = paper.get("pdf_url")

        if not pdf_url:
            continue

        temp_pdf = os.path.join(TEMP_DIR, f"temp_{task}_{i}.pdf")

        # --------------------------
        # Download PDF
        # --------------------------
        try:
            r = requests.get(pdf_url, timeout=30)
            if r.status_code != 200:
                continue
            with open(temp_pdf, "wb") as f:
                f.write(r.content)
        except:
            continue

        # --------------------------
        # Send to GROBID
        # --------------------------
        try:
            with open(temp_pdf, "rb") as f:
                r = requests.post(
                    GROBID_URL,
                    files={"input": f}
                )
            os.remove(temp_pdf)
        except:
            if os.path.exists(temp_pdf):
                os.remove(temp_pdf)
            continue

        if r.status_code != 200:
            continue

        tei_xml = r.text

        # --------------------------
        # Extract text from TEI
        # --------------------------
        try:
            root = ET.fromstring(tei_xml)
        except:
            continue

        texts = []
        for elem in root.iter():
            if elem.text:
                cleaned = elem.text.strip()
                if cleaned:
                    texts.append(cleaned)

        text = " ".join(texts)

        # ==============================
        # 🔥 MEMORY SAFETY GUARD
        # ==============================
        if len(text) > 150000:
            text = text[:150000]

        # ==============================
        # Chunked Stanza Processing
        # ==============================
        sentences = []
        chunk_size = 3000

        for j in range(0, len(text), chunk_size):
            chunk = text[j:j+chunk_size]

            try:
                doc = nlp(chunk)
                for sent in doc.sentences:
                    sentence_text = " ".join([token.text for token in sent.tokens])
                    sentences.append(sentence_text)
            except:
                continue

        # --------------------------
        # Triplet Extraction (SVO heuristic)
        # --------------------------
        rows = []

        for idx, sentence in enumerate(sentences):

            words = sentence.split()

            if len(words) >= 3:
                sub = words[0]
                pred = words[1]
                obj = " ".join(words[2:5])

                rows.append({
                    "topic": task,
                    "paper_ID": paper_id,
                    "sentence_ID": idx,
                    "info-unit": "auto",
                    "sub": sub,
                    "pred": pred,
                    "obj": obj,
                    "triplets": f"{sub} {pred} {obj}",
                    "pred_weights": None,
                    "year": year
                })

        # --------------------------
        # Incremental CSV Write
        # --------------------------
        if rows:
            df = pd.DataFrame(rows)
            df.to_csv(
                output_path,
                mode="a",
                header=not os.path.exists(output_path),
                index=False
            )

        # --------------------------
        # Save checkpoint (index only)
        # --------------------------
        with open(checkpoint_path, "w") as f:
            json.dump({"index": i + 1}, f)

    # ==========================
    # Cleanup checkpoint
    # ==========================
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    print(f"{task} Blogs triplets completed.")

In [9]:
for task in TASKS:
    build_blogs_triplets(task)

Dia Blogs triplets already completed.
MT Blogs triplets already completed.
NLI Blogs triplets already completed.


100%|██████████| 120/120 [1:57:48<00:00, 58.91s/it]  


Par Blogs triplets completed.


100%|██████████| 120/120 [1:10:42<00:00, 35.35s/it]


QA Blogs triplets completed.


100%|██████████| 120/120 [1:09:37<00:00, 34.81s/it]


SA Blogs triplets completed.


100%|██████████| 120/120 [1:39:29<00:00, 49.75s/it]  

Sum Blogs triplets completed.
